# ResQNet — Google Colab training notebook

Fine-tune **XLM-RoBERTa** for multilingual incident-category classification. The expanded dataset contains 6,480 synthetic template-generated examples in English, Hindi and Telugu. It is for pipeline practice only; native speakers must review language quality and labels before any research claim.

The workbook `data/ResQNet_Incident_Dataset.xlsx` contains Train, Validation and Test sheets. The matching CSV partitions are used below. Splits are grouped by simulated event to reduce near-duplicate leakage.

Before using real data: check dataset licence/terms, obtain appropriate permissions, have native-language reviewers validate labels, remove personal data, and preserve an untouched test set.

In [ ]:
# 1. Select Runtime > Change runtime type > T4 GPU (if available)
import torch, os, pandas as pd, numpy as np
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — training will be slow')

## 2. Upload the project ZIP
Upload `ResQNet.zip`. It contains the expanded Excel workbook and the three CSV partitions used for training, validation and testing.

In [ ]:
from google.colab import files
import zipfile, io, os
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload ResQNet.zip')
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    z.extractall('/content')
PROJECT_DIR = '/content/ResQNet'
TRAIN_PATH = f'{PROJECT_DIR}/data/train_reports.csv'
VAL_PATH = f'{PROJECT_DIR}/data/validation_reports.csv'
TEST_PATH = f'{PROJECT_DIR}/data/test_reports.csv'
assert all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]), 'Dataset split CSV files are missing'
print('Project ready:', PROJECT_DIR)

In [ ]:
# 3. Install model-training dependencies
!pip -q install -U "transformers>=4.46" datasets accelerate scikit-learn pandas


In [ ]:
# 4. Inspect the training, validation and test partitions
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
required = {'text','language','category','event_group'}
for name, frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    assert required.issubset(frame.columns), f'{name} missing columns: {required - set(frame.columns)}'
    print(name, frame.shape, 'synthetic rows:', int(frame.synthetic.astype(bool).sum()))
display(pd.crosstab(train_df.language, train_df.category))
# Confirm that event groups do not overlap between partitions.
assert set(train_df.event_group).isdisjoint(val_df.event_group)
assert set(train_df.event_group).isdisjoint(test_df.event_group)
assert set(val_df.event_group).isdisjoint(test_df.event_group)
print('No event_group leakage across splits.')

## 5. Create a smoke-test split
This small synthetic corpus is not large enough for reliable model evaluation. The following stratified split only tests whether training and export work. For research, use event/source-held-out splits and do not tune against the final test set.

In [ ]:
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['category'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['category'])
print('Train / validation / test:', len(train_df), len(val_df), len(test_df))

In [ ]:
# 6. Encode labels and tokenize multilingual text
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
df = pd.concat([train_df, val_df, test_df], ignore_index=True)
le = LabelEncoder().fit(train_df['category'])
label2id = {label: int(i) for i, label in enumerate(le.classes_)}
id2label = {int(i): label for label, i in label2id.items()}
def to_dataset(frame):
    out = frame[['text','category']].copy()
    out['labels'] = le.transform(out.category)
    return Dataset.from_pandas(out[['text','labels']], preserve_index=False)
datasets = DatasetDict({'train':to_dataset(train_df), 'validation':to_dataset(val_df), 'test':to_dataset(test_df)})
MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)
tokenized = datasets.map(tokenize, batched=True)

In [ ]:
# 7. Fine-tune XLM-R. If CUDA runs out of memory, reduce batch size to 4.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le.classes_), id2label=id2label, label2id=label2id
)
def compute_metrics(prediction):
    y_true = prediction.label_ids
    y_pred = np.argmax(prediction.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'accuracy':accuracy_score(y_true,y_pred), 'macro_precision':precision, 'macro_recall':recall, 'macro_f1':f1}
args = TrainingArguments(
    output_dir='/content/resqnet-xlmr-checkpoints',
    eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=3, weight_decay=0.01, load_best_model_at_end=True,
    metric_for_best_model='macro_f1', report_to='none',
    fp16=torch.cuda.is_available(), save_total_limit=2
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'],
                  eval_dataset=tokenized['validation'], compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# 8. Evaluate once on the held-out smoke-test set
test_metrics = trainer.evaluate(tokenized['test'])
print(test_metrics)
print('Synthetic-data metrics are not evidence of real-world quality.')

In [ ]:
# 9. Export model, tokenizer, label map and metrics
import json
OUT = '/content/resqnet-xlmr-model'
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
with open(f'{OUT}/label_mapping.json','w',encoding='utf-8') as f:
    json.dump({'label2id':label2id,'id2label':id2label,'metrics':test_metrics}, f, ensure_ascii=False, indent=2)
!cd /content && zip -qr resqnet-xlmr-model.zip resqnet-xlmr-model
files.download('/content/resqnet-xlmr-model.zip')

## 10. Team handoff
1. Store the exported model artifact in the team's agreed shared storage; large weights usually should not be committed to Git.
2. Record data source/licence, dataset version, split method, model revision, hyperparameters and metrics in the model card.
3. Add a dedicated inference adapter in `backend/engine.py` to load the fine-tuned model. The current app intentionally continues using its baseline until that adapter is implemented and tested.
4. Evaluate per-language and per-class precision/recall, confusion matrix, latency, calibration and out-of-domain examples on a real held-out set.
5. Never use synthetic-data metrics to claim emergency-response accuracy.